Data ingestion into Bronze layer


In [0]:
spark.sql("SELECT current_catalog(), current_database()").show()

# See available catalogs
spark.sql("SHOW CATALOGS").show()

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS olist_bronze")
spark.sql("USE olist_bronze")
spark.sql("SELECT current_database()").show()

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/olist_bronze/csvs/"))

In [0]:
from pyspark.sql.functions import current_timestamp, lit

vol_path = "/Volumes/workspace/olist_bronze/csvs"

def read_csv(filename):
    return spark.read\
        .option("header" ,"true")\
        .option("inferschema", "true")\
        .option("multiline", "true")\
        .option("escape" , '"')\
        .csv(f"{vol_path}/{filename}")
def load_to_bronze(df, table_name):
    df =df.withColumn("_ingest_at", current_timestamp())\
        .withColumn("_source", lit(table_name))
    df.writeTo(f"workspace.olist_bronze.{table_name}") \
      .using("delta") \
      .createOrReplace()
    count = spark.table(f"workspace.olist_bronze.{table_name}").count()
    print(f" {table_name}: {count:,} rows")
print("Setup done")


In [0]:
tables = {
    "orders":        "olist_orders_dataset.csv",
    "order_items":   "olist_order_items_dataset.csv",
    "customers":     "olist_customers_dataset.csv",
    "products":      "olist_products_dataset.csv",
    "sellers":       "olist_sellers_dataset.csv",
    "payments":      "olist_order_payments_dataset.csv",
    "reviews":       "olist_order_reviews_dataset.csv",
    "geolocation":   "olist_geolocation_dataset.csv",
    "category_map":  "product_category_name_translation.csv",
}

for table_name, filename in tables.items():
    print(f"Ingesting: {table_name}...")
    df = read_csv(filename)
    load_to_bronze(df, table_name)

print("\nBronze layer complete!")

In [0]:
print("=== Bronze Layer Health Check ===\n")

for table_name in tables.keys():
    df = spark.table(f"workspace.olist_bronze.{table_name}")
    print(f"{table_name}: {df.count():,} rows | {len(df.columns)} cols")